# UALF lightning: parse and fetch examples

This notebook shows how to:
- Configure credentials for the Frost API (met.no)
- Fetch lightning data in UALF format for an area of interest
- Parse UALF text into structured Python objects
- Optionally convert results to a pandas DataFrame for exploration

> Note: replace the coordinates with your own area of interest if desired.

In [9]:
# Use ualf client + .env credentials
from dotenv import load_dotenv
from ualf import fetch_lightning_events, to_dataframe

# Ensure .env is loaded for FROST_CLIENT_ID
load_dotenv(override=False)

# Configure area of interest (lat, lon)
lat_lon = (60.0, 10.0)  # replace with your own coordinates if needed
max_age = "P1D"  # ISO-8601 duration, e.g., P1D for 1 day

# Fetch events (returns list[Lightning])
events = fetch_lightning_events(lat_lon, size=2.0, max_age=max_age, referencetime="latest")

# Convert to DataFrame when pandas is installed (otherwise returns list-of-dicts)
df = to_dataframe(events)
df if hasattr(df, "head") else (df[:3] if isinstance(df, list) else df)

""


## Authentication

The Frost API requires a client id.
A client id can be obtained from [frost.met.no](https://frost.met.no/auth/requestCredentials.html).

Create a `.env` file in the project root:

```
FROST_CLIENT_ID=your-id
```

The notebook uses `python-dotenv` to load it automatically.

In [ ]:
# Example: build URL and fetch raw UALF text using the client helper
from ualf import build_url, create_geojson_polygon

lat_lon = (60.0, 10.0)  # Example coordinates
size = 1.0 # Size in degrees
max_age = "P1D"
ref_time = "latest"

requests_url = build_url(lat_lon, size, max_age, ref_time)

'https://frost.met.no/lightning/v0.ualf?referencetime=latest&geometry=POLYGON((9.0 59.0, 11.0 59.0, 11.0 61.0, 9.0 61.0, 9.0 59.0))&maxage=P1D'

### Request parameters

- `lat_lon`: center coordinate (lat, lon)
- `size`: half-size of the square in degrees (1.0 ≈ 1°)
- `max_age`: ISO-8601 duration (e.g., `P1D` for 1 day back)
- `referencetime`: use `latest` or a specific timestamp
- Geometry is sent as a WKT polygon to the Frost API.

## Parsing UALF text

Below we parse a single UALF line to show the structure of the resulting `Lightning` object and its dictionary representation.

### UALF field layout
![UALF](UALF_format.png)

Layout image courtesy of: [frost.met.no](https://frost.met.no/dataclarifications.html#ualf)

In [ ]:
# Import the parser from the module
from ualf import Lightning, parse_ualf_line, parse_ualf, to_dataframe

# Keep a sample for quick manual checks (matches README example)
sample = "0 2025 01 01 12 00 00 000000000 64.0000 10.0000 22 0 20 33 117.17 0.23 0.11 0.97 16.8 14.6 2.8 0 1 0 1"

In [12]:
# Demo: parse the provided example response line using the module
try:
    evt = parse_ualf_line(sample)
    print("Lightning dataclass:")
    print(evt)
    print("\nAs dict:")
    print(evt.to_dict())
except Exception as e:
    print("Failed to parse sample:", e)

Lightning dataclass:
Lightning(version=0, timestamp=datetime.datetime(2025, 1, 1, 12, 0, tzinfo=datetime.timezone.utc), lat=64.0, lon=10.0, peak_current=22, multiplicity=0, sensors=20, degrees_of_freedom=33, angle=117.17, semi_major_axis=0.23, semi_minor_axis=0.11, chi_square=0.97, rise_time=16.8, peak_to_zero_time=14.6, max_rate_of_rise=2.8, cloud_indicator=False, angle_indicator=True, signal_indicator=False, timing_indicator=True)

As dict:
{'version': 0, 'timestamp': '2025-01-01T12:00:00+00:00', 'lat': 64.0, 'lon': 10.0, 'peak_current': 22, 'multiplicity': 0, 'sensors': 20, 'degrees_of_freedom': 33, 'angle': 117.17, 'semi_major_axis': 0.23, 'semi_minor_axis': 0.11, 'chi_square': 0.97, 'rise_time': 16.8, 'peak_to_zero_time': 14.6, 'max_rate_of_rise': 2.8, 'cloud_indicator': False, 'angle_indicator': True, 'signal_indicator': False, 'timing_indicator': True}
